# Experiment C-4


In [1]:
"""C-4: MNIST handwritten digit detection using PyTorch, Keras, and TensorFlow."""

import numpy as np


def limit_data(x_train, y_train, x_test, y_test, train_n=10000, test_n=2000):
    return (
        x_train[:train_n],
        y_train[:train_n],
        x_test[:test_n],
        y_test[:test_n],
    )


def run_keras_mnist(x_train, y_train, x_test, y_test):
    try:
        import tensorflow as tf
    except ImportError:
        print("TensorFlow/Keras not installed. Skipping Keras section.")
        return None

    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1)),
            tf.keras.layers.Conv2D(16, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Conv2D(32, 3, activation="relu"),
            tf.keras.layers.MaxPooling2D(),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(10, activation="softmax"),
        ]
    )
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(x_train, y_train, epochs=1, batch_size=128, verbose=0)
    _, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"Keras CNN Accuracy: {acc * 100:.2f}%")
    return acc * 100


def run_tensorflow_mnist(x_train, y_train, x_test, y_test):
    try:
        import tensorflow as tf
    except ImportError:
        print("TensorFlow not installed. Skipping TensorFlow section.")
        return None

    class TFModel(tf.Module):
        def __init__(self):
            super().__init__()
            self.w1 = tf.Variable(tf.random.normal([784, 128], stddev=0.1))
            self.b1 = tf.Variable(tf.zeros([128]))
            self.w2 = tf.Variable(tf.random.normal([128, 10], stddev=0.1))
            self.b2 = tf.Variable(tf.zeros([10]))

        def __call__(self, x):
            x = tf.reshape(x, [-1, 784])
            h = tf.nn.relu(tf.matmul(x, self.w1) + self.b1)
            return tf.matmul(h, self.w2) + self.b2

    model = TFModel()
    optimizer = tf.keras.optimizers.Adam(0.001)

    x_train_tf = tf.convert_to_tensor(x_train, dtype=tf.float32)
    y_train_tf = tf.convert_to_tensor(y_train, dtype=tf.int32)
    x_test_tf = tf.convert_to_tensor(x_test, dtype=tf.float32)
    y_test_tf = tf.convert_to_tensor(y_test, dtype=tf.int32)

    batch_size = 128
    n = x_train_tf.shape[0]
    for _ in range(1):
        for i in range(0, n, batch_size):
            xb = x_train_tf[i : i + batch_size]
            yb = y_train_tf[i : i + batch_size]
            with tf.GradientTape() as tape:
                logits = model(xb)
                loss = tf.reduce_mean(
                    tf.nn.sparse_softmax_cross_entropy_with_logits(labels=yb, logits=logits)
                )
            grads = tape.gradient(loss, [model.w1, model.b1, model.w2, model.b2])
            optimizer.apply_gradients(zip(grads, [model.w1, model.b1, model.w2, model.b2]))

    logits_test = model(x_test_tf)
    preds = tf.argmax(logits_test, axis=1, output_type=tf.int32)
    acc = tf.reduce_mean(tf.cast(tf.equal(preds, y_test_tf), tf.float32)).numpy() * 100
    print(f"TensorFlow (low-level) Accuracy: {acc:.2f}%")
    return acc


def run_pytorch_mnist(x_train, y_train, x_test, y_test):
    try:
        import torch
        import torch.nn as nn
        import torch.optim as optim
        from torch.utils.data import DataLoader, TensorDataset
    except ImportError:
        print("PyTorch not installed. Skipping PyTorch section.")
        return None

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    x_tr = torch.tensor(x_train, dtype=torch.float32).permute(0, 3, 1, 2)
    y_tr = torch.tensor(y_train, dtype=torch.long)
    x_te = torch.tensor(x_test, dtype=torch.float32).permute(0, 3, 1, 2)
    y_te = torch.tensor(y_test, dtype=torch.long)

    train_loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=128, shuffle=True)
    test_loader = DataLoader(TensorDataset(x_te, y_te), batch_size=256)

    model = nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * 5 * 5, 64),
        nn.ReLU(),
        nn.Linear(64, 10),
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    acc = (correct / total) * 100
    print(f"PyTorch CNN Accuracy: {acc:.2f}%")
    return acc


def main() -> None:
    try:
        from tensorflow.keras.datasets import mnist
    except Exception:
        print("TensorFlow/Keras dataset loader unavailable. Install tensorflow first.")
        return

    (x_train, y_train), (x_test, y_test) = mnist.load_data()
    x_train = (x_train.astype(np.float32) / 255.0)[..., None]
    x_test = (x_test.astype(np.float32) / 255.0)[..., None]

    x_train, y_train, x_test, y_test = limit_data(x_train, y_train, x_test, y_test)

    print("MNIST subset: train=10000, test=2000")
    print("Running Keras, TensorFlow(low-level), and PyTorch models...\n")

    k_acc = run_keras_mnist(x_train, y_train, x_test, y_test)
    t_acc = run_tensorflow_mnist(x_train, y_train, x_test, y_test)
    p_acc = run_pytorch_mnist(x_train, y_train, x_test, y_test)

    print("\nSummary (accuracy %):")
    print(f"Keras: {k_acc if k_acc is not None else 'N/A'}")
    print(f"TensorFlow: {t_acc if t_acc is not None else 'N/A'}")
    print(f"PyTorch: {p_acc if p_acc is not None else 'N/A'}")


if __name__ == "__main__":
    main()



ModuleNotFoundError: No module named 'numpy'